# Trackerless Freehand US 2D→3D — End-to-End Kaggle Runner

This notebook clones the repo, installs it, and runs the **entire pipeline** end-to-end.

It has two parts:

- **Part A — Synthetic pipeline (always runs):** Stages 0→4 (encoder → pose → compounding →
  implicit field → render) plus the 4-metric evaluation, all on tiny synthetic data. This is the
  guaranteed-green smoke path and needs no dataset attached.
- **Part B — Real TUS-REC data (auto-detected):** Runs the encoder + official 4-metric evaluation on a
  real attached TUS-REC dataset. This section **only activates if** (a) a dataset is attached and (b) the
  real-data loader / `stage_eval` code is present in the cloned branch. Otherwise it prints guidance and
  skips — it never crashes the notebook.

Every stage writes `outputs/<stage>/manifest.json` (pass/fail + shapes) and figures to
`outputs/figures/<stage>/`. Part C prints a summary table from those manifests.

> **To use real data:** click **+ Add Data** in the Kaggle sidebar, attach the TUS-REC dataset, then set
> `DATASET_SLUG` in the config cell to its `/kaggle/input/<slug>` folder name.

## 0. Clone + install

In [ ]:
# --- Clone the repository (idempotent) ---
import os

REPO_URL  = "https://github.com/Armstrong66/2D-to-3D-medical-image-reconstruction.git"
REPO_DIR  = "2D-to-3D-medical-image-reconstruction"
# Branch that holds the real-data loader + 4-metric eval work.
# Use "main" if you only want the synthetic pipeline (Part A).
BRANCH    = "feat/real-data-loader-eval-metrics"

base = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(base)
if not os.path.isdir(REPO_DIR):
    os.system(f"git clone -q --branch {BRANCH} {REPO_URL} || git clone -q {REPO_URL}")
os.chdir(REPO_DIR)
# make sure we are on the requested branch if it exists on the remote
os.system(f"git checkout -q {BRANCH} 2>/dev/null || true")
print("cwd:", os.getcwd())
os.system("git log --oneline -1")

In [ ]:
# --- Install the package (editable) + h5py, and make imports resolve on Kaggle ---
import sys, site, subprocess
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=False)
# h5py is needed by the real-data loader; install explicitly in case requirements lag.
subprocess.run([sys.executable, "-m", "pip", "install", "h5py", "-q"], check=False)

site.main()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    src_dir = candidate / "src"
    if (src_dir / "usrecon").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        print(f"[usrecon] added {src_dir} to sys.path")
        break

# drop any cached usrecon modules so the fresh install is picked up
for m in [k for k in list(sys.modules) if k == "usrecon" or k.startswith("usrecon.")]:
    del sys.modules[m]
import usrecon
print("usrecon imported from:", usrecon.__file__)

## 1. Environment + config

In [ ]:
# --- Report the execution environment ---
import torch, platform
from usrecon.utils.device import resolve_device

print("python  :", platform.python_version())
print("torch   :", torch.__version__)
print("cuda    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu     :", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"vram    : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

In [ ]:
# --- Load config and apply Kaggle-friendly overrides ---
from usrecon.pipeline.run_stage import _load_config

cfg = _load_config("src/usrecon/config/default.yaml")

# A slightly longer stage-3 schedule so the loss curve is meaningful on a real run.
cfg["training"]["epochs"] = 100
cfg["device"]["gpus"] = "auto"      # auto | cpu | 0 | 0,1

# Set this to the attached dataset folder under /kaggle/input to enable Part B.
# e.g. DATASET_SLUG = "tus-rec-2024"  ->  /kaggle/input/tus-rec-2024
DATASET_SLUG = ""

print("config ready. epochs =", cfg["training"]["epochs"],
      "| image_size =", cfg["data"]["image_size"])

In [ ]:
# --- Capability detection: which real-data / eval pieces exist in this checkout? ---
import importlib
import usrecon.pipeline.run_stage as _rs

def have(module_path, attr=None):
    try:
        mod = importlib.import_module(module_path)
        return (attr is None) or hasattr(mod, attr)
    except Exception:
        return False

HAS_STAGE_EVAL   = hasattr(_rs, "run_stage_eval")
HAS_REAL_LOADER  = have("usrecon.data.datasets", "TUSRECScanDataset")
HAS_METRICS      = have("usrecon.eval.metrics", "reconstruction_errors")

print("run_stage_eval      :", HAS_STAGE_EVAL)
print("TUSRECScanDataset   :", HAS_REAL_LOADER)
print("reconstruction_errors:", HAS_METRICS)
if not (HAS_STAGE_EVAL and HAS_REAL_LOADER and HAS_METRICS):
    print("\n[note] Real-data loader / eval not fully present on this branch yet.")
    print("       Part A (synthetic) runs fully; Part B will self-skip.")

## Part A — Synthetic pipeline (always runs)

Runs every stage on tiny synthetic data. No dataset required. Each cell prints an `... OK` line and
saves figures; `show_latest` displays them inline.

In [ ]:
# Stage 0 — Encoder (frame -> embedding)
from usrecon.pipeline.run_stage import run_stage0_encoder
from usrecon.utils.viz import show_latest
run_stage0_encoder(cfg, use_synthetic=True)
show_latest(stage="stage0_encoder")

In [ ]:
# Stage 1 — Pose / motion estimation
from usrecon.pipeline.run_stage import run_stage1_pose
run_stage1_pose(cfg, use_synthetic=True)
show_latest(stage="stage1_pose")

In [ ]:
# Stage 2 — Compounding (frames + poses -> point cloud)
from usrecon.pipeline.run_stage import run_stage2_compounding
run_stage2_compounding(cfg, use_synthetic=True)

In [ ]:
# Stage 3 — Implicit neural volume (coordinate-MLP fit)
from usrecon.pipeline.run_stage import run_stage3_implicit_field
run_stage3_implicit_field(cfg, use_synthetic=True)
show_latest(stage="stage3_implicit_field")

In [ ]:
# Stage 4 — Render the frozen field on a 3D grid
from usrecon.pipeline.run_stage import run_stage4_render
run_stage4_render(cfg, use_synthetic=True)

In [ ]:
# 4-metric evaluation on synthetic data (self-skips if stage_eval not present yet)
if HAS_STAGE_EVAL:
    _rs.run_stage_eval(cfg, use_synthetic=True)
    show_latest(stage="stage_eval")
else:
    print("[skip] run_stage_eval not in this checkout — implement the plan's Task 5 to enable it.")

## Part B — Real TUS-REC data (auto-detected)

Activates only when a dataset is attached (`DATASET_SLUG` set or something under `/kaggle/input`) **and**
the real-data loader + eval code are present. It points the config at the dataset root and runs the
encoder + official 4-metric evaluation on real sweeps.

In [ ]:
# --- Locate the attached real dataset root ---
from pathlib import Path

def find_dataset_root():
    # 1) explicit slug
    if DATASET_SLUG:
        p = Path("/kaggle/input") / DATASET_SLUG
        if p.exists():
            return p
    # 2) helper that knows the configured names
    try:
        from usrecon.data import get_dataset_path
        p = get_dataset_path(cfg["data"].get("dataset_name", ""))
        if p:
            return Path(p)
    except Exception:
        pass
    # 3) first non-empty folder under /kaggle/input
    ki = Path("/kaggle/input")
    if ki.exists():
        for child in sorted(ki.iterdir()):
            if child.is_dir() and any(child.rglob("*.h5")):
                return child
    return None

DATA_ROOT = find_dataset_root()
print("resolved real-data root:", DATA_ROOT)
if DATA_ROOT is not None:
    cfg["data"]["root"] = str(DATA_ROOT)
    n_h5 = len(list(Path(DATA_ROOT).rglob("*.h5")))
    print(f"found {n_h5} .h5 scan files under the root")

In [ ]:
# --- Run the encoder on real frames (guarded) ---
REAL_READY = (DATA_ROOT is not None) and HAS_REAL_LOADER
if REAL_READY:
    try:
        run_stage0_encoder(cfg, use_synthetic=False)
        show_latest(stage="stage0_encoder")
    except Exception as e:
        print("[stage0 real-data] failed:", repr(e))
        print("This usually means the on-disk layout differs from the loader's expectation —")
        print("check <subject>/<scan>.h5 with datasets 'frames' and 'tforms'.")
else:
    print("[skip] real-data encoder — need both an attached dataset and TUSRECScanDataset.")

In [ ]:
# --- Official 4-metric evaluation on real data (guarded) ---
if REAL_READY and HAS_STAGE_EVAL and HAS_METRICS:
    try:
        _rs.run_stage_eval(cfg, use_synthetic=False)
        show_latest(stage="stage_eval")
    except Exception as e:
        print("[stage_eval real-data] failed:", repr(e))
else:
    print("[skip] real-data eval — requires attached data + loader + metrics + stage_eval.")
    print("       Implement the plan (docs/superpowers/plans/...) then re-run this cell.")

## Part C — Results summary

Reads every `outputs/<stage>/manifest.json` and prints status + key fields. This is the same manifest
trail used for the “send me the traceback” debugging loop.

In [ ]:
# --- Summarize all stage manifests ---
import json
from usrecon.paths import OUTPUT_DIR

rows = []
for man in sorted(OUTPUT_DIR.glob("*/manifest.json")):
    try:
        d = json.loads(man.read_text())
    except Exception as e:
        rows.append((man.parent.name, "unreadable", repr(e)))
        continue
    stage = man.parent.name
    status = d.get("status", "?")
    # pull a few interesting keys if present
    highlights = {k: d[k] for k in (
        "output_shape", "loss_total", "final_loss", "num_points",
        "voxel_shape", "global_all", "local_all",
        "global_landmark", "local_landmark", "num_scans") if k in d}
    rows.append((stage, status, highlights))

print(f"{'stage':<24} {'status':<8} details")
print("-" * 80)
for stage, status, hl in rows:
    print(f"{stage:<24} {status:<8} {hl}")
if not rows:
    print("no manifests found — did the stages run?")